
# Credit Risk Scoring Framework: Baseline Model

## Project Goal

This notebook builds a **simple, interpretable credit risk scoring framework** using the German Credit dataset.

The goal is to construct a **rule-based risk score** that reflects basic credit risk intuition and can later be used for:

- sensitivity analysis
- scenario analysis
- stress testing

This design is useful for demonstrating how borrower characteristics can be translated into a structured risk framework in a transparent and explainable way.

---

## Why Use a Rule-Based Framework Instead of Machine Learning?

In many credit risk and model validation settings, interpretability is extremely important.  
A rule-based scoring framework has several advantages:

1. **Transparency**  
   Each variable enters the score in a visible and explainable way.

2. **Economic intuition**  
   The direction of each variable can be justified using common credit risk logic.

3. **Scenario flexibility**  
   Once a score is constructed, it is easy to test how risk changes when borrower conditions deteriorate.

4. **Validation value**  
   This framework is especially useful for showing how risk responds under controlled changes, which is central to sensitivity analysis and stress testing.

---

## Conceptual Setup

The notebook follows a simple idea:

### Step 1: Map borrower characteristics into numeric risk components

Some variables are categorical, such as:

- Saving accounts
- Checking account
- Housing

These variables are converted into ordered numeric scores so that they can enter a risk formula.

For example:

- stronger savings position → lower risk
- stronger checking balance → lower risk
- owning a home → lower risk than renting

### Step 2: Standardize continuous variables

Continuous variables such as:

- Age
- Credit amount
- Duration

are standardized so they are on a more comparable scale.

### Step 3: Construct an interpretable risk score

A weighted score is created using borrower characteristics.

The sign of each term is chosen based on basic credit risk intuition:

- **higher credit amount** → more risk
- **longer duration** → more risk
- **older age** → slightly less risk
- **more savings / checking balance** → less risk
- **more stable housing / job status** → less risk

### Step 4: Transform the score into a pseudo probability of default

The risk score is converted into a bounded value between 0 and 1 using a logistic transformation.

This value is called **pseudo PD** in this notebook.

It should be interpreted as:

> a risk-oriented probability-style measure derived from an interpretable score

rather than a formally estimated default probability from a statistical model.

---

## Important Note

This notebook does **not** estimate a regulatory or production-grade PD model.

Instead, it builds a **demonstration risk scoring framework** for analytical purposes.

Therefore:

- the weights are **manually chosen**
- the pseudo PD is **illustrative**
- the emphasis is on **interpretability and scenario usefulness**, not predictive optimization

This is intentional, because the broader project focuses on:

- model logic
- borrower risk sensitivity
- scenario-based validation

---

## Variables Used in the Scoring Framework

The baseline score uses the following inputs:

### Continuous variables
- **Age**
- **Credit amount**
- **Duration**

### Categorical variables converted into scores
- **Saving accounts**
- **Checking account**
- **Housing**
- **Job**

### Variables not directly used in the baseline score
- **Sex**
- **Purpose**

These variables are retained in the dataset for descriptive analysis, but they are not included in the main score in order to keep the initial framework simpler and easier to interpret.

---

## Expected Risk Direction

The score is designed so that the following patterns hold:

- larger loans should increase risk
- longer repayment periods should increase risk
- weaker liquidity conditions should increase risk
- weaker housing status should increase risk
- weaker job category should increase risk

If the resulting pseudo PD patterns align with these expectations, that supports the internal consistency of the scoring framework.

---

## Notebook Outputs

At the end of this notebook, the following outputs will be produced:

1. **Pseudo PD summary statistics**
2. **Risk bucket distribution**
3. **Top high-risk borrowers**
4. **Average pseudo PD by borrower segments**, such as:
   - Housing
   - Purpose
   - Saving accounts

These outputs provide the baseline reference point for the next notebooks on:

- sensitivity analysis
- stress testing

---

## Interpretation

This baseline notebook should be read as answering the following question:

> If we assign interpretable risk weights to borrower characteristics, what does the resulting borrower risk distribution look like?

The next step will be to ask:

> How does this distribution change when borrower conditions worsen?

That is where sensitivity analysis and stress testing become useful.

---

In [3]:
import pandas as pd
import numpy as np

# ======================
# 1. Read data
# ======================
df = pd.read_csv("/Users/quentingao/Desktop/german_credit_data.csv")

# Drop useless index column
if "Unnamed: 0" in df.columns:
    df = df.drop(columns=["Unnamed: 0"])

print("Columns:")
print(df.columns.tolist())
print("\nShape:", df.shape)
print(df.head())
# ======================
# 2. Basic cleaning
# ======================
# Keep missing values explicit for now
df = df.copy()

# Standardize text columns a bit
for col in df.select_dtypes(include="object").columns:
    df[col] = df[col].astype(str).str.strip()

# Convert common string missing placeholders back to np.nan
df = df.replace({"nan": np.nan, "NA": np.nan})

# ======================
# 3. Map categorical risk-related variables into scores
# Higher score = safer
# ======================

# Saving accounts
saving_map = {
    "little": 0,
    "moderate": 1,
    "quite rich": 2,
    "rich": 3
}
df["saving_score"] = df["Saving accounts"].map(saving_map)

# Missing savings treated conservatively but not worst-case
df["saving_score"] = df["saving_score"].fillna(0.5)

# Checking account
checking_map = {
    "little": 0,
    "moderate": 1,
    "rich": 2
}
df["checking_score"] = df["Checking account"].map(checking_map)
df["checking_score"] = df["checking_score"].fillna(0.5)

# Housing
housing_map = {
    "rent": 0,
    "free": 1,
    "own": 2
}
df["housing_score"] = df["Housing"].map(housing_map)

# Job
# Assuming larger value = more skilled / stable
# If your coding means something else, we can revise later
df["job_score"] = df["Job"]

# Sex / Purpose are left out at first to keep the framework cleaner
# and avoid unnecessary demographic interpretation

# ======================
# 4. Standardize continuous variables manually
# We want a simple interpretable scoring formula
# ======================
continuous_vars = ["Age", "Credit amount", "Duration"]

for col in continuous_vars:
    mean_col = df[col].mean()
    std_col = df[col].std()
    df[f"{col}_z"] = (df[col] - mean_col) / std_col

# ======================
# 5. Construct an interpretable risk score
# Higher score = riskier
# ======================
# Economic intuition:
# - Higher credit amount -> riskier
# - Longer duration -> riskier
# - Older age -> slightly safer
# - More savings/checking balance -> safer
# - Better housing/job status -> safer

df["risk_score"] = (
    0.60 * df["Credit amount_z"]
    + 0.75 * df["Duration_z"]
    - 0.25 * df["Age_z"]
    - 0.80 * df["saving_score"]
    - 0.60 * df["checking_score"]
    - 0.30 * df["housing_score"]
    - 0.20 * df["job_score"]
)

# ======================
# 6. Convert risk score into pseudo probability of default (PD)
# Logistic transformation
# ======================
df["pseudo_pd"] = 1 / (1 + np.exp(-df["risk_score"]))

# ======================
# 7. Create risk buckets
# ======================
df["risk_bucket"] = pd.cut(
    df["pseudo_pd"],
    bins=[0, 0.20, 0.40, 0.60, 1.00],
    labels=["Low", "Moderate", "Elevated", "High"],
    include_lowest=True
)

# ======================
# 8. Summary output
# ======================
print("\nPseudo PD summary:")
print(df["pseudo_pd"].describe())

print("\nRisk bucket distribution:")
print(df["risk_bucket"].value_counts(dropna=False))

# ======================
# 9. Inspect highest-risk borrowers
# ======================
cols_to_show = [
    "Age", "Sex", "Job", "Housing", "Saving accounts", "Checking account",
    "Credit amount", "Duration", "Purpose",
    "risk_score", "pseudo_pd", "risk_bucket"
]

print("\nTop 10 riskiest borrowers:")
print(df[cols_to_show].sort_values("pseudo_pd", ascending=False).head(10))

# ======================
# 10. Segment analysis
# ======================
print("\nAverage pseudo PD by Housing:")
print(df.groupby("Housing")["pseudo_pd"].mean().sort_values(ascending=False))

print("\nAverage pseudo PD by Purpose:")
print(df.groupby("Purpose")["pseudo_pd"].mean().sort_values(ascending=False))

print("\nAverage pseudo PD by Saving accounts:")
print(df.groupby("Saving accounts")["pseudo_pd"].mean().sort_values(ascending=False))

Columns:
['Age', 'Sex', 'Job', 'Housing', 'Saving accounts', 'Checking account', 'Credit amount', 'Duration', 'Purpose']

Shape: (1000, 9)
   Age     Sex  Job Housing Saving accounts Checking account  Credit amount  \
0   67    male    2     own             NaN           little           1169   
1   22  female    2     own          little         moderate           5951   
2   49    male    1     own          little              NaN           2096   
3   45    male    2    free          little           little           7882   
4   53    male    2    free          little           little           4870   

   Duration              Purpose  
0         6             radio/TV  
1        48             radio/TV  
2        12            education  
3        42  furniture/equipment  
4        24                  car  

Pseudo PD summary:
count    1000.000000
mean        0.237174
std         0.225776
min         0.004137
25%         0.077399
50%         0.154343
75%         0.327882
max      

# Results Summary and Interpretation

## Distribution of Pseudo PD

The pseudo probability of default (PD) shows a reasonable distribution across borrowers:

- Mean PD is approximately **0.24**, indicating a moderate overall risk level in the sample.
- The distribution is **right-skewed**, with most borrowers concentrated in lower-risk regions.
- A small portion of borrowers exhibit relatively high PD values, representing higher-risk profiles.

This pattern is consistent with typical credit datasets, where the majority of borrowers are relatively safe, and only a subset carries elevated risk.

---

## Risk Segmentation

Borrowers are divided into four risk buckets:

- **Low risk**
- **Moderate risk**
- **Elevated risk**
- **High risk**

The distribution across these buckets provides a simple segmentation of the portfolio.  
Such segmentation is useful for:

- prioritizing monitoring efforts
- guiding lending decisions
- supporting downstream risk reporting

---

## Relationship with Financial Characteristics

The pseudo PD aligns well with economic intuition:

### Savings and Liquidity

- Borrowers with **higher savings levels** (e.g., "rich", "quite rich") exhibit **lower PD**
- Borrowers with **little or missing savings information** tend to have **higher PD**

This suggests that liquidity is a key driver of credit risk in the scoring framework.

---

### Loan Size and Duration

- Larger **credit amounts** contribute positively to the risk score
- Longer **loan durations** are associated with higher PD

These relationships reflect standard credit risk logic:

> larger exposure and longer repayment horizons increase uncertainty and default risk

---

### Housing Status

- Borrowers who **own their homes** tend to have lower PD
- Those who **rent** or have weaker housing status tend to show higher risk

Housing acts as a proxy for financial stability and long-term wealth.

---

## Internal Consistency Check

Overall, the scoring framework produces results that are **directionally consistent with expected credit risk behavior**:

- stronger financial position → lower risk  
- weaker financial indicators → higher risk  

This supports the internal validity of the constructed risk score.

---

## Limitations

It is important to emphasize that:

- The model is **not statistically estimated**
- The weights are **manually specified**
- The pseudo PD is **not calibrated to real default probabilities**

Therefore, the output should be interpreted as:

> a structured and interpretable risk representation rather than a predictive model

---

## Next Step

This baseline framework will serve as the foundation for:

- **Sensitivity Analysis**  
  Evaluating how PD changes when individual variables are adjusted

- **Stress Testing**  
  Evaluating how the portfolio behaves under adverse scenarios

These analyses will help assess how borrower risk responds to changes in financial conditions.